In [5]:
import sys
import yaml
import torch
sys.path.append("..")  # Adds the project root, so 'src' is importable
from src.model.echoingecg_model import EchoingECG

# load model

In [ ]:
with open("src/configs/model.yaml") as f:
    model_cfg = yaml.safe_load(f)

In [3]:
model_cfg

{'ecg_encoder': {'numleads': 12,
  'init_dim': 96,
  'factor_dim': [2, 4, 8, 16],
  'embed_size': 512,
  'var_norm': False,
  'mean_norm': True},
 'text_encoder': {'embed_size': 512,
  'var_norm': False,
  'mean_norm': True,
  'gpo_dim': 32},
 'embed_size': 512}

In [4]:
model = EchoingECG(model_cfg)

pytorch_model.bin:   0%|          | 0.00/433M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

In [ ]:
# load echoingecg.pt
model_weights = torch.load("echoingecg.pt", weights_only=True, map_location="cpu")
model.load_state_dict(model_weights)

<All keys matched successfully>

# ecg example

In [8]:
dummy_ecg = torch.zeros((1, 12,1000)) #ecg is 10 seconds at 100hz, 12 lead
input = {"ecg": dummy_ecg}

In [12]:
output = model(input)

In [ ]:
output.keys() #model will have the same keys as input, takes in 'text' and 'ecg'

dict_keys(['ecg'])

In [ ]:
output["ecg"].keys() #has a mean and std (probabilistic)

dict_keys(['mean', 'std'])

In [17]:
output["ecg"]["mean"].shape, output["ecg"]["std"].shape

(torch.Size([1, 512]), torch.Size([1, 512]))

# text example

In [18]:
from transformers import AutoTokenizer

In [23]:
tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-v1.1",return_pt=True)

In [35]:
text_example = "ecg is normal"
tok_dict = tokenizer(text_example)
input_model = {"text": torch.tensor(tok_dict["input_ids"]).unsqueeze(0), "attention_mask": torch.tensor(tok_dict["attention_mask"]).unsqueeze(0)}

In [37]:
output = model(input_model)

In [38]:
output["text"].keys() #has a mean and std (probabilistic)

dict_keys(['mean', 'std'])

In [39]:
output["text"]["mean"].shape, output["text"]["std"].shape

(torch.Size([1, 512]), torch.Size([1, 512]))

# load an ECG properly

In [43]:
from src.datasets.helpers import scale_ecg
import joblib
import numpy as np

In [ ]:
# load ecg_scaler.pkl
sc = joblib.load("ecg_scaler.pkl")
_center = torch.from_numpy(sc.mean_.astype(np.float32))  # shape (L,)
_scale = torch.from_numpy(sc.scale_.astype(np.float32)).clamp_min(1e-8)  # (L,)

In [ ]:
dummy_ecg = torch.zeros((1,12,1000)) # batch, leads, length
scaled_output = scale_ecg(_center, _scale, dummy_ecg) #please use our scaler provided